### Pre-training의 첫번째 관문
- 데이터의 역활 : 다다익선 
- 데이터처리 파이프라인
    - 원본 텍스트 데이터
        - 1단계 : 필터링(스팸, 저품질 제거)
        - 2단계 : 중복제거
        - 3단계 : 정규화
        - 4단계 : 토큰화
    - 훈련가능한 데이터
- 토큰화 : 텍스트를 모델이 이해할수 있는 정수 시퀀스로 변환하는 과정
    - 1. 전처리
        - 공백표준화
        - 특수문자처리
        - 케이스통일
    - 2. 어휘선택
        - Character-level : 문자단위(너무 길어짐)
        - Word-level : 단어단위(어휘크기 폭증)
        - Subword-level : n-gram (GPT 계열) --> 가장 효율적
    - 3. 어휘사전
        - 가장 자주나타나는 시퀀스 학습(playing = play + ing)
- Byte-Pair Encoding(BPE)
    - 1. 초기 단어 분해
        hello word -> [h, e , l ....]
    - 2. 가장 빈번한 바이트 쌍 병합
        반복 1 : ('l', 'l') -> 'll'
        반복 2 : ('h', 'e') -> 'he'
        반복 3 : ('w', 'o') -> 'wo'
        ...
    - 3. 병합규칙 저장
        merge_rules = {
            ('l','l') : 'B1'
            ...
        }
    - 4. 어휘생성(단어사전 vocabulary)
        vocab = ['h','e',,, 'll','he']

#### BPE의 장점
1. **어휘 크기 조절 가능**: 50K ~ 200K 범위에서 유연함
2. **미지의 단어 처리**: OOV(Out-of-Vocabulary) 거의 발생하지 않음
3. **효율성**: 긴 문장을 적정 길이로 압축

### 4. 토큰화가 성능에 미치는 영향

#### 예시 1: "PlayGround"
- **나쁜 토큰화**: ['P', 'l', 'a', 'y', 'G', 'r', 'o', 'u', 'n', 'd'] (10개)
- **좋은 토큰화**: ['Play', 'Ground'] (2개)
- **영향**: 입력 길이 감소 → 모델이 더 넓은 맥락 볼 수 있음

#### 예시 2: 다국어 처리
- 영어: 효율적 토큰화 가능
- 한국어/중국어: 문자 단위에 가까워야 함 (언어별 토크나이저 필요)

![image.png](attachment:image.png)

In [31]:
from collections import Counter, defaultdict
import re
from typing import List, Dict, Tuple
import pandas as pd
import matplotlib.pyplot as plt
# 재현성
import random
random.seed(42)

In [32]:
# 작은 말뭉치
corpus = [
    "hello world hello there",
    "hello beautiful world",
    "the quick brown fox",
    "the lazy dog",
    "hello hello",
    "world peace"
]
class BPETokenizer:
    '''
    BPE(Byte-Pair-Encoding)

    작동:
    1. 텍스트를 문자 단위로 분해
    2. 가장 자주 나타나는 인접한 쌍(pair)을 찾음
    3. 그 쌍을 하나의 새 토큰으로 병합
    4. 2-3을 반복해서 어휘크기를 조절
    '''
    def __init__(self):
        self.word_freqs = defaultdict(int) # 단어빈도
        self.merges = {}  # 병합규칙(pair) -> new token
        self.vocab = set() # 현재 어휘
        self.merge_history = []  # 병합과정 기록
    def _build_initial_vocab(self, corpus:List[str]):
        '''step 1 : 텍스트를 단어어로 분해하고 문자 단위로 분리'''
        for text in corpus:
            words = text.split()
            for word in words:
                self.word_freqs[word] += 1
        
        # 초기어휘 : 모든 문자
        for word in self.word_freqs:
            for char in word:
                self.vocab.add(char)
        self.vocab.add('</w>')  # 특수토큰 : 단어 끝을 표시(GPT 스타일)
        return self
    
    def get_stats(self, vocab:Dict[str,int]) -> Dict[Tuple[str,str], int]:
        '''각(pair) 조합의 빈도를 계산'''
        pairs = defaultdict(int)
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols)-1):
                pairs[symbols[i], symbols[i+1]] += freq              
        return pairs
    
    def merge_vocab(self, pair:Tuple[str,str], vocab:Dict[str,int]) -> Dict[str,int]:
        '''특정 pair를 하나의 토큰으로 병합'''
        new_vocab = {}
        bigram =' '.join(pair)
        replacement = ''.join(pair)
        for word in vocab:
            new_word = word.replace(bigram, replacement)
            new_vocab[new_word] = vocab[word]
        return new_vocab
    
    def train(self, corpus: List[str], num_merges: int = 10):
        """BPE 훈련: num_merges 번만큼 반복"""
        self._build_initial_vocab(corpus)
        
        # 초기 vocab dict 생성
        vocab = {}
        for word, freq in self.word_freqs.items():
            #  각 문자 사이에 공백 추가
            vocab[' '.join(word) + ' </w>'] = freq
        
        print(f"[초기 어휘 크기] {len(self.vocab)}")
        print(f"[병합 시작] {num_merges}번 반복\n")
        
        for i in range(num_merges):
            pairs = self.get_stats(vocab)
            
            if not pairs:
                print(f"  반복 {i+1}: 병합 가능한 pair가 없음. 중단.")
                break
            
            best_pair = max(pairs, key=pairs.get)
            best_freq = pairs[best_pair]
            
            # 각 병합 단계 상세 기록
            vocab = self.merge_vocab(best_pair, vocab)
            self.merges[best_pair] = ''.join(best_pair)
            new_token = ''.join(best_pair)
            self.vocab.add(new_token)
            
            merge_record = {
                'iteration': i + 1,
                'pair': f"('{best_pair[0]}', '{best_pair[1]}')",
                'frequency': best_freq,
                'new_token': new_token,
                'vocab_size': len(self.vocab)
            }
            self.merge_history.append(merge_record)
            
            print(f"  반복 {i+1}: {best_pair} → '{new_token}' (빈도: {best_freq})")
        
        print(f"\n[최종 어휘 크기] {len(self.vocab)}")
        return self

In [33]:
tokenizer = BPETokenizer()
tokenizer.train(corpus=corpus, num_merges=10)

[초기 어휘 크기] 23
[병합 시작] 10번 반복

  반복 1: ('h', 'e') → 'he' (빈도: 8)
  반복 2: ('he', 'l') → 'hel' (빈도: 5)
  반복 3: ('hel', 'l') → 'hell' (빈도: 5)
  반복 4: ('hell', 'o') → 'hello' (빈도: 5)
  반복 5: ('hello', '</w>') → 'hello</w>' (빈도: 5)
  반복 6: ('w', 'o') → 'wo' (빈도: 3)
  반복 7: ('wo', 'r') → 'wor' (빈도: 3)
  반복 8: ('wor', 'l') → 'worl' (빈도: 3)
  반복 9: ('worl', 'd') → 'world' (빈도: 3)
  반복 10: ('world', '</w>') → 'world</w>' (빈도: 3)

[최종 어휘 크기] 33


In [34]:
tokenizer.merges

{('h', 'e'): 'he',
 ('he', 'l'): 'hel',
 ('hel', 'l'): 'hell',
 ('hell', 'o'): 'hello',
 ('hello', '</w>'): 'hello</w>',
 ('w', 'o'): 'wo',
 ('wo', 'r'): 'wor',
 ('wor', 'l'): 'worl',
 ('worl', 'd'): 'world',
 ('world', '</w>'): 'world</w>'}

In [35]:
def tokenize_word(word: str, merges: dict, vocab: set) -> List[str]:
    """학습된 merge 규칙을 적용하여 단어 토큰화"""
    # Step 1: 문자 단위로 분해
    chars = list(word) + ['</w>']

    # Step 2: merge 규칙을 순서대로 적용
    for pair, new_token in merges.items():
        while True:
            found = False
            for j in range(len(chars) - 1):
                if chars[j] == pair[0] and chars[j + 1] == pair[1]:
                    chars[j:j + 2] = [new_token]
                    found = True
                    break
            if not found:
                break

    return chars

print("=" * 70)
print("[토큰화 시연]")
print("=" * 70)

#  실제 단어로 테스트
test_words = ["hello", "world", "beautiful", "the", "peace"]

for word in test_words:
    tokens = tokenize_word(word, tokenizer.merges, tokenizer.vocab)
    print(f"\n'{word}' →")

    # 단계별 분해 과정
    chars_initial = list(word) + ['</w>']
    print(f"  1. 초기: {chars_initial}")
    print(f"  2. 최종: {tokens}")
    print(f"  3. 토큰 개수: {len(chars_initial)} → {len(tokens)} (압축: {len(chars_initial) - len(tokens)}개 감소)")

[토큰화 시연]

'hello' →
  1. 초기: ['h', 'e', 'l', 'l', 'o', '</w>']
  2. 최종: ['hello</w>']
  3. 토큰 개수: 6 → 1 (압축: 5개 감소)

'world' →
  1. 초기: ['w', 'o', 'r', 'l', 'd', '</w>']
  2. 최종: ['world</w>']
  3. 토큰 개수: 6 → 1 (압축: 5개 감소)

'beautiful' →
  1. 초기: ['b', 'e', 'a', 'u', 't', 'i', 'f', 'u', 'l', '</w>']
  2. 최종: ['b', 'e', 'a', 'u', 't', 'i', 'f', 'u', 'l', '</w>']
  3. 토큰 개수: 10 → 10 (압축: 0개 감소)

'the' →
  1. 초기: ['t', 'h', 'e', '</w>']
  2. 최종: ['t', 'he', '</w>']
  3. 토큰 개수: 4 → 3 (압축: 1개 감소)

'peace' →
  1. 초기: ['p', 'e', 'a', 'c', 'e', '</w>']
  2. 최종: ['p', 'e', 'a', 'c', 'e', '</w>']
  3. 토큰 개수: 6 → 6 (압축: 0개 감소)


In [37]:
#  여러 규모로 토크나이저 훈련
results = []
merge_counts = [0, 5, 10, 15, 20]

test_sentence = "hello world hello beautiful"
test_words_in_sentence = test_sentence.split()

print("="*70)
print("[어휘 크기의 영향]")
print(f"테스트 문장: '{test_sentence}'")
print("="*70)

for num_merges in merge_counts:
    tok = BPETokenizer()
    
    # 조용히 훈련 (print 억제)
    tok._build_initial_vocab(corpus)
    vocab = {}
    for word, freq in tok.word_freqs.items():
        vocab[' '.join(word) + ' </w>'] = freq
    
    pairs = tok.get_stats(vocab)
    for i in range(num_merges):
        if not pairs:
            break
        best_pair = max(pairs, key=pairs.get)
        vocab = tok.merge_vocab(best_pair, vocab)
        tok.merges[best_pair] = ''.join(best_pair)
        tok.vocab.add(''.join(best_pair))
        pairs = tok.get_stats(vocab)
    
    # 테스트 문장 토큰화
    total_tokens = 0
    tokenization_details = {}
    for word in test_words_in_sentence:
        tokens = tokenize_word(word, tok.merges, tok.vocab)
        total_tokens += len(tokens)
        tokenization_details[word] = tokens
    
    result = {
        'Merge_횟수': num_merges,
        '어휘_크기': len(tok.vocab),
        '총_토큰_수': total_tokens,
        '압축율': f"{(len(test_sentence) - total_tokens) / len(test_sentence) * 100:.1f}%"
    }
    results.append(result)
    
    #  각 케이스의 토큰화 결과
    print(f"\n[Merge 횟수: {num_merges}]")
    print(f"  어휘 크기: {len(tok.vocab)}")
    for word in test_words_in_sentence:
        tokens = tokenization_details[word]
        print(f"    '{word}' → {tokens}")
    print(f"  전체: 총 {total_tokens} 토큰 (원본 {len(test_sentence)} 문자)")

# 비교 테이블
print("\n" + "="*70)
print("[비교 분석]")
print("="*70)
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

[어휘 크기의 영향]
테스트 문장: 'hello world hello beautiful'

[Merge 횟수: 0]
  어휘 크기: 23
    'hello' → ['h', 'e', 'l', 'l', 'o', '</w>']
    'world' → ['w', 'o', 'r', 'l', 'd', '</w>']
    'hello' → ['h', 'e', 'l', 'l', 'o', '</w>']
    'beautiful' → ['b', 'e', 'a', 'u', 't', 'i', 'f', 'u', 'l', '</w>']
  전체: 총 28 토큰 (원본 27 문자)

[Merge 횟수: 5]
  어휘 크기: 28
    'hello' → ['hello</w>']
    'world' → ['w', 'o', 'r', 'l', 'd', '</w>']
    'hello' → ['hello</w>']
    'beautiful' → ['b', 'e', 'a', 'u', 't', 'i', 'f', 'u', 'l', '</w>']
  전체: 총 18 토큰 (원본 27 문자)

[Merge 횟수: 10]
  어휘 크기: 33
    'hello' → ['hello</w>']
    'world' → ['world</w>']
    'hello' → ['hello</w>']
    'beautiful' → ['b', 'e', 'a', 'u', 't', 'i', 'f', 'u', 'l', '</w>']
  전체: 총 13 토큰 (원본 27 문자)

[Merge 횟수: 15]
  어휘 크기: 38
    'hello' → ['hello</w>']
    'world' → ['world</w>']
    'hello' → ['hello</w>']
    'beautiful' → ['b', 'ea', 'u', 't', 'i', 'f', 'u', 'l', '</w>']
  전체: 총 12 토큰 (원본 27 문자)

[Merge 횟수: 20]
  어휘 크기: 43
    'hello' 